# معمل التجربة والتطبيق العملي: المسترجع الهجين في LangChain
## Playground: Combining Dense and Sparse Retriever with LangChain (`Lecture 42`)

مرحباً بك في المعمل العملي للمحاضرة 42 من مسار **RAG Mastery**.

---

### أهداف المعمل:
1. تجربة بناء مسترجع كثيف ومسترجع متناثر بشكل مستقل.
2. استخدام `EnsembleRetriever` لدمج النتائج واختبار تأثير تغيير الأوزان (`weights`).
3. تجربة استعلامات دلالية (`Semantic Paraphrase`) واستعلامات حرفية (`Exact Acronym / Code`).
4. ربط المسترجع الهجين بسلسلة RAG كاملة للحصول على إجابة نهائية.


In [ ]:
# 1. إعداد البيئة واستيراد المكتبات
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers.bm25 import BM25Retriever

try:
    from langchain.retrievers.ensemble import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers.ensemble import EnsembleRetriever

try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains import create_retrieval_chain
except ImportError:
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

print("[INFO] المكتبات جاهزة للعمل!")


In [ ]:
# 2. تجهيز المستندات وإنشاء المسترجعين (الكثيف والمتناثر)
docs = [
    Document(page_content="LangChain helps build LLM applications", metadata={"source": "doc1"}),
    Document(page_content="LangChain can be used to develop Agentic AI applications", metadata={"source": "doc2"}),
    Document(page_content="LangChain has many types of retrievers", metadata={"source": "doc3"}),
    Document(page_content="Pinecone is a vector database for semantic search", metadata={"source": "doc4"}),
    Document(page_content="The Eiffel Tower is located in Paris and is a famous tourist attraction", metadata={"source": "doc5"})
]

# Dense Retriever
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Sparse Retriever
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3

print(f"[INFO] تم بناء الفهرس الكثيف والمتناثر لـ {len(docs)} مستندات.")


---
### مساحة التجربة والتطبيق العملي 1: مقارنة أداء المسترجعين بشكل منفرد

**المطلوب:**
قم بتجربة استعلامين مختلفين:
1. استعلام دلالي بحت (لا توجد به كلمات مطابقة حرفياً، مثل: `"tools to construct intelligent reasoning systems"`).
2. استعلام حرفي دقيق (يحتوي على كلمة محددة جداً مثل: `"retrievers"`).
لاحظ الفرق في ترتيب النتائج بين `dense_retriever` و `sparse_retriever`.


In [ ]:
# مساحة التجربة والتطبيق العملي 1
# اكتب كودك هنا لمقارنة نتائج dense_retriever و sparse_retriever
query_semantic = "tools to construct intelligent reasoning systems"
query_lexical = "retrievers"

# جرب invoke على كل مسترجع واطبع النتائج:
# dense_results = dense_retriever.invoke(query_semantic)
# sparse_results = sparse_retriever.invoke(query_semantic)


---
### مساحة التجربة والتطبيق العملي 2: اختبار تأثير تغيير أوزان EnsembleRetriever

**المطلوب:**
1. قم بإنشاء `EnsembleRetriever` بثلاثة توزيعات مختلفة للأوزان:
   - النموذج المتوازن: `weights=[0.5, 0.5]`
   - تفضيل البحث الدلالي: `weights=[0.85, 0.15]`
   - تفضيل البحث الحرفي: `weights=[0.15, 0.85]`
2. اختبر استعلاماً دقيقاً ولاحظ كيف يتغير ترتيب المستندات المسترجعة.


In [ ]:
# مساحة التجربة والتطبيق العملي 2
# جرب إنشاء 3 مسترجعات هجينة بأوزان مختلفة وقارن النتائج:

# hybrid_balanced = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.5, 0.5])
# hybrid_dense_heavy = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.85, 0.15])
# hybrid_sparse_heavy = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.15, 0.85])

# query = "vector database"
# اطبع نتائج كل مسترجع


---
### مساحة التجربة والتطبيق العملي 3: بناء RAG Pipeline مخصص بالكامل

**المطلوب:**
قم بإضافة مستندين جديدين من تخصصك (مثلاً عن الطب أو المالية أو هندسة البرمجيات)، وأعد بناء المسترجع الهجين، ثم اطلب من النموذج اللغوي الإجابة على سؤال يتطلب الجمع بين معلومتين.


In [ ]:
# مساحة التجربة والتطبيق العملي 3
# اكتب كودك هنا لإضافة المستندات وبناء سلسلة الـ RAG وتوليد الإجابة
